In [ ]:
# 1. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
# 2. 드라이브 내에 안전한 작업 폴더 생성
SAFE_DIR = '/content/drive/MyDrive/pothole_project'
os.makedirs(f'{SAFE_DIR}/weights', exist_ok=True)
os.makedirs(f'{SAFE_DIR}/dataset', exist_ok=True)
os.makedirs(f'{SAFE_DIR}/runs', exist_ok=True) # YOLO의 학습 결과가 실시간으로 저장
print("구글 드라이브 작업 폴더 및 runs 폴더 생성 완료")

# 3. GPU 할당 확인
!nvidia-smi

# 4. Ultralytics(YOLOv11) 설치 및 환경 검증
!pip install ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY_HERE")
project = rf.workspace("jeongsus-workspace").project("pothole_without_augment-s6ovi")
version = project.version(2)
dataset = version.download("yolov11")

In [ ]:
import os

# 1. 방금 다운로드 성공한 dataset 객체에게 진짜 위치를 물어봅니다.
source_dir = dataset.location
print(f"👀 현재 데이터가 숨어있는 곳: {source_dir}")

# 2. 우리의 안전한 구글 드라이브 금고 경로
target_dir = "/content/drive/MyDrive/pothole_project/dataset"
os.makedirs(target_dir, exist_ok=True)

# 3. 리눅스 강제 복사 명령어로 구글 드라이브에 안전하게 밀어 넣습니다.
print("🚀 구글 드라이브로 데이터를 안전하게 수송 중입니다...")
!cp -r "{source_dir}/"* "{target_dir}/"

# 4. 수송 완료 후 최종 확인!
print("\n✅ 수송 완료! 구글 드라이브 금고 안의 파일 목록:")
!ls -al "{target_dir}"

In [ ]:
# YOLOv11n (Nano) 모델을 사용하여 30번(Epochs) 반복 학습을 진행합니다.
!yolo task=detect mode=train model=yolo11n.pt data=/content/drive/MyDrive/pothole_project/dataset/data.yaml epochs=30 imgsz=640 batch=16 project=/content/drive/MyDrive/pothole_project/runs name=baseline_run

In [ ]:
import cv2
import time
import glob
from ultralytics import YOLO

# 1. 방금 구워낸 우리의 소중한 모델 로드
model_path = "/content/drive/MyDrive/pothole_project/runs/baseline_run/weights/best.pt"
model = YOLO(model_path)

# 2. 테스트용 이미지 하나 무작위로 가져오기 (test 폴더에서)
test_images = glob.glob("/content/drive/MyDrive/pothole_project/dataset/test/images/*.jpg")
sample_image_path = test_images[0]
frame = cv2.imread(sample_image_path)

# 3. [백엔드 A 빙의] 우리가 만든 run_inference 로직 실행!
print("🚀 백엔드 A가 이미지를 전송했습니다. AI 추론 시작...\n")
start_time = time.time()
results = model.predict(source=frame, conf=0.4, imgsz=640, verbose=False)
inference_time_ms = round((time.time() - start_time) * 1000, 2)

detections = []
for box in results[0].boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    conf = float(box.conf[0])
    detections.append({
        "class": "pothole",
        "confidence": round(conf, 2),
        "bbox": [int(x1), int(y1), int(x2), int(y2)]
    })

# 4. 최종 출력 결과 (이 JSON이 백엔드 A에게 날아갑니다!)
output_data = {
    "frame_timestamp": 1710557515.123,  # 가짜 타임스탬프
    "device_id": "test_cam_01",         # 가짜 기기 ID
    "inference_time_ms": inference_time_ms,
    "detections": detections
}

import json
print("📦 [최종 반환 JSON 데이터]")
print(json.dumps(output_data, indent=2, ensure_ascii=False))